# Cálculo de R vs C para MNIST (Autocontenido)

**Objetivo:** Calcular el parámetro de orden R en función del parámetro de acoplamiento C en el rango [0, 2.0] para todo el dataset MNIST.

**Metodología:**
- Para cada imagen del dataset
- Evaluar R(C) en el rango C ∈ [0, 2.0] con 200 puntos
- Guardar los arrays completos de R y C en base de datos SQLite

**Estructura de la DB:**
- Una tabla por clase (clase_0 a clase_9)
- Columnas: `id`, `image_idx`, `c_values` (BLOB), `r_values` (BLOB), `timestamp`, `seed`

**Optimizaciones:**
- Soporte MPS (Apple M3), CUDA y CPU
- 200 puntos C (rango completo, idéntico a calculo_alpha_c)
- Persistencia en Google Drive

## 1. Imports y Configuración

In [ ]:
import os
import sys
import sqlite3
import pickle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm.notebook import tqdm
from datetime import datetime
import matplotlib.pyplot as plt
from pathlib import Path
import time
import random

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if hasattr(torch.backends, 'mps'):
    print(f"MPS available: {torch.backends.mps.is_available()}")

## 1.1. Montar Google Drive (Solo Colab)

**IMPORTANTE**: En Google Colab, los datos se guardarán en Google Drive para persistencia entre sesiones.

In [ ]:
# Detectar si estamos en Colab y montar Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Configurar rutas en Google Drive
    DRIVE_ROOT = Path('/content/drive/MyDrive/analisis_criticalidad')
    DRIVE_ROOT.mkdir(exist_ok=True, parents=True)
    
    print("✅ Google Drive montado correctamente")
    print(f"📁 Directorio de trabajo: {DRIVE_ROOT}")
    
    IN_COLAB = True
    
except ImportError:
    print("ℹ️  No estás en Colab, trabajando en directorio local")
    DRIVE_ROOT = Path(".")
    IN_COLAB = False

## 1.2. Verificar Base de Datos Existente

Si ya iniciaste el procesamiento anteriormente, esta celda mostrará el progreso guardado.

In [ ]:
# Verificar si existe base de datos previa en Drive
db_check_path = DRIVE_ROOT / "resultados_R_vs_C" / "mnist_R_vs_C_v2.db"  # ← NUEVO NOMBRE

if db_check_path.exists():
    print("✅ BASE DE DATOS EXISTENTE ENCONTRADA")
    print(f"📁 Ubicación: {db_check_path}")
    print(f"💾 Tamaño: {db_check_path.stat().st_size / (1024*1024):.2f} MB")
    
    # Mostrar progreso por clase
    try:
        conn = sqlite3.connect(str(db_check_path))
        cursor = conn.cursor()
        
        print("\n📊 Progreso guardado:")
        total = 0
        for clase in range(10):
            try:
                cursor.execute(f'SELECT COUNT(*) FROM clase_{clase}')
                count = cursor.fetchone()[0]
                total += count
                if count > 0:
                    print(f"   Clase {clase}: {count:,} imágenes")
            except:
                pass
        
        conn.close()
        print(f"\n   Total: {total:,} imágenes procesadas")
        print("\n🔄 El procesamiento continuará desde donde se quedó")
        
    except Exception as e:
        print(f"⚠️  Error al leer DB: {e}")
else:
    print("🆕 No se encontró base de datos previa")
    print("   Se creará una nueva base de datos")

## 2. Configuración de Semillas para Reproducibilidad

In [ ]:
# Semilla fija para reproducibilidad (IGUAL que calculo_alpha_c_mnist_autocontenido.ipynb)
SEED = 1

def set_seed(seed):
    """Fija todas las semillas para reproducibilidad completa"""
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
print(f"✅ Semilla configurada: {SEED}")
print(f"   Esta es la MISMA semilla que calculo_alpha_c_mnist_autocontenido.ipynb")

## 3. Detección de Dispositivo (GPU/CPU)

In [ ]:
# Detectar dispositivo (CUDA, MPS o CPU)
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f"🚀 Usando GPU: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    print(f"🚀 Usando Apple Silicon (MPS)")
else:
    DEVICE = torch.device('cpu')
    print(f"💻 Usando CPU")

print(f"Dispositivo seleccionado: {DEVICE}")

## 4. Definición del Modelo Kuramoto (Autocontenido)

**Modelo copiado del módulo para que el notebook sea completamente autocontenido**

In [ ]:
import einops

def reshape(x, n):
    return einops.rearrange(x, 'b (c n) ... -> b c n ...', n=n)

def reshape_back(x):
    return einops.rearrange(x, 'b c n ... -> b (c n) ...')

class ModReLU(nn.Module):
    def __init__(self, n, ch, norm="gn"):
        super().__init__()
        self.n = n
        if norm == "bn":
            self.norm = nn.BatchNorm2d(ch // n)
        elif norm == "gn":
            self.norm = nn.GroupNorm(ch // n, ch // n)
        else:
            self.norm = nn.Identity()

    def forward(self, x):
        x = reshape(x, self.n)
        m = torch.linalg.norm(x, dim=2)
        m = torch.nn.ReLU()(self.norm(m))
        x = m.unsqueeze(2) * torch.nn.functional.normalize(x, dim=2)
        x = reshape_back(x)
        return x

class KConv2d(nn.Module):
    def __init__(self, n, ch, connectivity='conv', ksize=3, init_omg=1.0, hw=(16,16), use_omega=True, use_omega_c=True):
        super().__init__()
        assert (ch % n) == 0
        self.n = n
        self.ch = ch

        if connectivity == 'conv':
            self.connectivity = nn.Conv2d(ch, ch, ksize, 1, ksize//2, bias=False)
        elif connectivity == 'conv_mlp':
            self.connectivity = nn.Sequential(
                nn.Conv2d(ch, ch, ksize, 1, ksize//2, bias=False),
                ModReLU(n, ch),
                nn.Conv2d(ch, ch, ksize, 1, ksize//2, bias=False))
        else:
            raise NotImplementedError

        self.use_omega = use_omega
        self.use_omega_c = use_omega_c
        if use_omega or use_omega_c:
            if n == 2:
                self.omg_param = nn.Parameter(torch.randn(ch//2, 2))
            else:
                self.omg_param = nn.Parameter(init_omg * (1/np.sqrt(n))* torch.randn(ch//n, n, n))

    def omg(self, p):
        if self.n==2:
            p = torch.linalg.norm(p, dim=1)
            return torch.stack(
                [torch.stack([torch.zeros_like(p), p], -1),
                torch.stack([-p, torch.zeros_like(p)], -1)],
                    -1)
        else:
            return p - p.transpose(1, 2)

    def forward(self, x, c=None):
        y = self.connectivity(x)
        if c is not None:
            y = y + c
        y = reshape(y, self.n)
        x = reshape(x, self.n)

        omg_x = torch.einsum('cnm,bcmhw->bcnhw', self.omg(self.omg_param), x) if self.use_omega else torch.zeros_like(x)
        proj = y - torch.sum(y*x, 2, keepdim=True) * x
        if c is not None:
            c = reshape(c, self.n)
            omg_c = torch.einsum('cnm,bcmhw->bcnhw', self.omg(self.omg_param), c) if self.use_omega_c else torch.zeros_like(c)
            return reshape_back(omg_x + proj), reshape_back(omg_c)
        else:
            return reshape_back(omg_x + proj)

    def compute_energy(self, x, c=None):
        y = self.connectivity(x)
        y = y + c
        B = x.shape[0]
        return - torch.sum(x.view(B, -1) * y.view(B, -1), -1)

class KBlock(nn.Module):
    def __init__(self, n, ch, connectivity='conv', T=4, ksize=3, init_omg=0.1, c_norm='gn', use_omega=True, use_omega_c=True):
        super().__init__()
        self.n = n
        self.ch = ch
        self.T = T
        self.kconv = KConv2d(n, ch, connectivity=connectivity, ksize=ksize, init_omg=init_omg, use_omega=use_omega, use_omega_c=use_omega_c)
        self.monitor_count = 0
        if c_norm == 'gn':
            self.c_norm = nn.GroupNorm(ch//n, ch)
        else:
            self.c_norm = lambda x: x

    def normalize(self, x, y=None):
        x = reshape(x, self.n)
        x = torch.nn.functional.normalize(x, dim=2)
        if y is not None:
            x = torch.linalg.norm(reshape(y, self.n), dim=2, keepdim=True) * x
        x = reshape_back(x)
        return x

    def forward(self, x, c, T, gamma, del_t=1.0, return_xs=False, return_es=False, T_noc=None):
        x = self.normalize(x)
        c = self.c_norm(c)
        xs = [x]
        es = []
        if return_es:
            energy = self.kconv.compute_energy(x, c)
            es.append(energy)

        for t in range(T):
            dxdt, dcdt = self.kconv(x, c)
            _c = c + gamma*del_t*dcdt
            c = self.normalize(_c, c)
            x = x + gamma*del_t*dxdt
            x = self.normalize(x)

            if return_es:
                energy = self.kconv.compute_energy(x, c)
                es.append(energy)

            if return_xs:
                xs.append(x)

        if return_es:
            return x, xs, es
        else:
            return x, xs

print("✅ Modelos KBlock, KConv2d y ModReLU definidos")

## 5. Función de Parámetro de Orden R

In [ ]:
def kuramoto_order_parameter(xs):
    """
    Calcula el parámetro de orden de Kuramoto R(t)
    
    Args:
        xs: Lista de estados [T, B, C, H, W] donde C son canales complejos (pares real/imag)
    
    Returns:
        np.array: Valores de R(t) para cada paso temporal
    """
    R_values = []
    
    for x in xs:
        # Extraer componentes real e imaginaria
        if x.ndim == 4:
            x = x[0]  # Tomar primer batch
        
        # Calcular fase de cada oscilador (canales 0,1 representan primer oscilador)
        theta = torch.atan2(x[1::2], x[0::2])
        
        # Parámetro de orden: magnitud del promedio de exp(i*theta)
        cos_theta = torch.cos(theta)
        sin_theta = torch.sin(theta)
        
        r_x = cos_theta.mean()
        r_y = sin_theta.mean()
        
        R = torch.sqrt(r_x**2 + r_y**2)
        R_values.append(R.item())
    
    return np.array(R_values)

print("✅ Función de parámetro de orden R definida")

## 6. Configuración de Parámetros

In [ ]:
# Parámetros del modelo (IDÉNTICOS a calculo_alpha_c_mnist_autocontenido.ipynb)
CH = 3  # ← Corregido: era 4, debe ser 3
N = 3   # ← Corregido: era 4, debe ser 3
T_STEPS = 30
T_MODEL = 4
KSIZE = 3  # ← Corregido: era 7, debe ser 3
INIT_OMG = 0.1
IMG_SIZE = 64

# Parámetros de acoplamiento C (IDÉNTICOS a calculo_alpha_c)
C_MIN = 0.0
C_MAX = 2.0  # ← Actualizado: era 0.4, ahora 2.0 como calculo_alpha_c
N_C_POINTS = 200  # ← Actualizado: era 50, ahora 200 como calculo_alpha_c
c_range = np.arange(C_MIN, C_MAX, 0.01)  # ← Mismo que calculo_alpha_c

# Configuración de procesamiento
BATCH_SIZE = 1

# Directorios - USAR GOOGLE DRIVE SI ESTAMOS EN COLAB
if IN_COLAB:
    RESULTS_DIR = DRIVE_ROOT / "resultados_R_vs_C"
    print("📁 Usando Google Drive para persistencia de datos")
else:
    RESULTS_DIR = Path("resultados_R_vs_C")
    print("📁 Usando directorio local")

RESULTS_DIR.mkdir(exist_ok=True, parents=True)
# CAMBIAR nombre de DB para evitar conflicto con la antigua
DB_PATH = RESULTS_DIR / "mnist_R_vs_C_v2.db"  # ← NUEVO NOMBRE

print("\n📌 CONFIGURACIÓN:")
print(f"   🔧 Parámetros del modelo:")
print(f"      - CH (canales): {CH}")
print(f"      - N (dimensión oscilador): {N}")
print(f"      - KSIZE (kernel): {KSIZE}")
print(f"      - T (pasos temporales): {T_STEPS}")
print(f"      - IMG_SIZE: {IMG_SIZE}x{IMG_SIZE}")
print(f"   📊 Parámetros de análisis:")
print(f"      - Rango C: [{C_MIN}, {C_MAX}]")
print(f"      - Puntos C: {N_C_POINTS}")
print(f"      - Espaciado C: {0.01}")
print(f"   💾 Base de datos: {DB_PATH}")
print(f"\n✅ Estos parámetros son IDÉNTICOS a calculo_alpha_c_mnist_autocontenido.ipynb")
print(f"   para garantizar comparabilidad COMPLETA entre R vs C y α_c")

# Verificar si existe base de datos previa
if DB_PATH.exists():
    print(f"\n✅ Base de datos existente encontrada: {DB_PATH.stat().st_size / (1024*1024):.2f} MB")
    print("   El procesamiento continuará desde donde se quedó")
else:
    print(f"\n🆕 Se creará nueva base de datos en: {DB_PATH}")

## 7. Funciones de Base de Datos SQLite

In [ ]:
def inicializar_db(db_path):
    """Crea la base de datos con una tabla por clase para guardar R vs C"""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # Crear tabla para cada clase (0-9)
    for clase in range(10):
        tabla = f"clase_{clase}"
        cursor.execute(f'''
            CREATE TABLE IF NOT EXISTS {tabla} (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                image_idx INTEGER NOT NULL UNIQUE,
                c_values BLOB NOT NULL,
                r_values BLOB NOT NULL,
                timestamp TEXT NOT NULL,
                seed INTEGER,
                n INTEGER,
                ch INTEGER,
                T INTEGER,
                gamma REAL,
                del_t REAL
            )
        ''')
    
    conn.commit()
    conn.close()
    print(f"✅ Base de datos creada: {db_path}")
    print(f"   📊 Estructura: 10 tablas (clase_0 a clase_9)")
    print(f"   🔑 Columnas: id, image_idx, c_values (BLOB), r_values (BLOB), timestamp, params")

def guardar_resultado_db(db_path, clase, image_idx, c_values, r_values, params):
    """Guarda arrays de C y R en la base de datos"""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    timestamp = datetime.now().isoformat()
    
    # Convertir arrays a bytes usando pickle
    c_blob = pickle.dumps(c_values)
    r_blob = pickle.dumps(r_values)
    
    cursor.execute(f'''
    INSERT OR REPLACE INTO clase_{clase} 
    (image_idx, c_values, r_values, timestamp, seed, n, ch, T, gamma, del_t)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', (image_idx, c_blob, r_blob, timestamp, 
          params['seed'], params['n'], params['ch'], 
          params['T'], params['gamma'], params['del_t']))
    
    conn.commit()
    conn.close()

def contar_procesadas(db_path):
    """Cuenta cuántas imágenes se han procesado por clase"""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    conteo = {}
    for clase in range(10):
        tabla = f"clase_{clase}"
        try:
            cursor.execute(f'SELECT COUNT(*) FROM {tabla}')
            conteo[clase] = cursor.fetchone()[0]
        except:
            conteo[clase] = 0
    
    conn.close()
    return conteo

# Inicializar DB
inicializar_db(DB_PATH)

## 8. Cargar Dataset MNIST

In [ ]:
# Transformaciones
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

# Cargar dataset de entrenamiento (60,000 imágenes)
# En Colab, los datos se descargan automáticamente
train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,  # Mantener orden para reproducibilidad
    num_workers=2  # 0 en Colab si hay problemas
)

print(f"✅ Dataset cargado:")
print(f"   - Total imágenes: {len(train_dataset):,}")
print(f"   - Imágenes por clase: ~{len(train_dataset)//10:,}")

# Verificar progreso actual
conteo_actual = contar_procesadas(DB_PATH)
total_procesadas = sum(conteo_actual.values())

print(f"\n📊 Progreso actual:")
print(f"   - Total procesadas: {total_procesadas:,}/{len(train_dataset):,}")
for clase, count in conteo_actual.items():
    print(f"   - Clase {clase}: {count:,} imágenes")

## 9. Inicializar Modelo y Estado Inicial

**CRÍTICO:** El estado inicial `x_init` se crea UNA SOLA VEZ y se reutiliza para TODAS las imágenes (igual que calculo_alpha_c)

In [ ]:
# Reiniciar semilla para consistencia (IGUAL que calculo_alpha_c)
set_seed(SEED)

# Crear modelo con parámetros IDÉNTICOS a calculo_alpha_c
kblock = KBlock(
    n=N, 
    ch=CH, 
    connectivity='conv',
    T=T_MODEL, 
    ksize=KSIZE,
    init_omg=INIT_OMG,
    c_norm=None,  # ← IMPORTANTE: None como en calculo_alpha_c
    use_omega=True,
    use_omega_c=False  # ← IMPORTANTE: False como en calculo_alpha_c
)
kblock = kblock.to(DEVICE)
kblock.eval()  # Modo evaluación (sin dropout, etc.)

# Estado inicial (se reutilizará para TODAS las imágenes)
# CRÍTICO: Este x_init es el MISMO para todas las imágenes del dataset
# NOTA: CH canales (no CH*2), el reshape interno manejará los osciladores complejos
x_init = torch.randn(1, CH, IMG_SIZE, IMG_SIZE)

print(f"✅ Modelo inicializado en {DEVICE}")
print(f"✅ Estado inicial creado con forma {x_init.shape}")
print(f"   ⚠️  IMPORTANTE: x_init se reutilizará para TODAS las imágenes")
print(f"   Parámetros: N={N}, CH={CH}, KSIZE={KSIZE}")
print(f"   use_omega_c=False (igual que calculo_alpha_c)")
print(f"   Parámetros del modelo: {sum(p.numel() for p in kblock.parameters()):,}")

## 10. Función para calcular R(C)

**Evalúa el parámetro de orden R para cada valor de C en el rango especificado**

In [ ]:
def calcular_R_vs_C(img, kblock, x_init, device, c_range, ch=4, h=64, w=64, T=30, gamma=0.7, del_t=0.9):
    """
    Calcula R(C) para una imagen en el rango de C especificado.
    
    Args:
        img: Imagen de entrada [1, H, W]
        kblock: Modelo KBlock
        x_init: Estado inicial [1, ch*2, h, w] - MISMO para todas las imágenes
        device: Dispositivo (cuda/mps/cpu)
        c_range: Array de valores de C a evaluar
        ch, h, w: Dimensiones de los canales y resolución
        T: Pasos temporales
        gamma, del_t: Parámetros de integración
    
    Returns:
        tuple: (c_values, r_values) - Arrays numpy con los valores de C y R
    """
    # Preprocesar imagen
    img_resized = transforms.functional.resize(img, [h, w])
    img_channels = img_resized.repeat(ch, 1, 1)
    img_channels = img_channels.to(device)
    
    R_final = []
    
    # Evaluar para cada valor de C
    for c_val in c_range:
        # CRÍTICO: x_init se clona para cada C, pero es el MISMO x_init base
        x = x_init.clone().to(device)
        c = img_channels.unsqueeze(0) * c_val
        
        with torch.no_grad():
            _, xs = kblock(
                x, c,
                T=T,
                gamma=gamma,
                del_t=del_t,
                return_xs=True
            )
        
        # Calcular parámetro de orden R(t)
        R = kuramoto_order_parameter(xs)
        R_final.append(R[-1])  # Tomar valor final (estado estacionario)
    
    return c_range, np.array(R_final)

print("✅ Función de cálculo R(C) definida")
print("   Sin cálculo de C_crítico, solo evalúa R para cada C")

## 11. Procesamiento del Dataset Completo

**⚠️ IMPORTANTE**: Esta celda procesa todas las 60,000 imágenes. En Colab con GPU puede tomar ~3-5 horas.

Patrón idéntico a calculo_alpha_c_mnist_autocontenido.ipynb

### 💾 Persistencia y Checkpoints

**En Google Colab:**
- ✅ La base de datos se guarda automáticamente en Google Drive
- ✅ Commit a la DB cada imagen procesada
- ✅ Los datos persisten entre sesiones (puedes detener y reiniciar)
- ✅ Si Colab se desconecta, simplemente reinicia desde la última imagen guardada

**Estrategia de recuperación:**
1. La DB guarda cada imagen inmediatamente
2. Al reiniciar el notebook, detecta automáticamente las imágenes ya procesadas
3. Continúa solo con las pendientes

In [ ]:
# Organizar por clases
imagenes_por_clase = {i: [] for i in range(10)}  # ← IMPORTANTE: Siempre 10 clases

for i in range(len(train_dataset)):
    _, label = train_dataset[i]
    # Convertir label a entero de Python (puede ser tensor)
    label = int(label) if hasattr(label, 'item') else int(label)
    imagenes_por_clase[label].append(i)

print("\nDistribución por clase:")
for clase, indices in imagenes_por_clase.items():
    print(f"  Clase {clase}: {len(indices)} imágenes")

# Seleccionar clases a procesar
CLASES_A_PROCESAR = range(10)  # Todas las clases
# CLASES_A_PROCESAR = [0, 1]  # ← Descomenta para procesar solo clases específicas
LIMITE_POR_CLASE = None  # None = todas las imágenes
# LIMITE_POR_CLASE = 100  # ← Descomenta para procesar solo 100 imágenes por clase (testing)

print(f"\n⚙️  Clases seleccionadas para procesar: {list(CLASES_A_PROCESAR)}")
if LIMITE_POR_CLASE:
    print(f"   Límite por clase: {LIMITE_POR_CLASE} imágenes")
else:
    print("   Procesando TODAS las imágenes de cada clase")

print("\n" + "="*70)
print("INICIANDO PROCESAMIENTO")
print("="*70)
print(f"🔐 REPRODUCIBILIDAD:")
print(f"   - Semilla global: {SEED}")
print(f"   - x_init: MISMO para TODAS las imágenes")
print(f"📊 DATOS GUARDADOS:")
print(f"   - Por imagen: {N_C_POINTS} valores de R y C")
print(f"   - Sin cálculo de C_crítico (solo R vs C)")
print("="*70 + "\n")

# PARAMS para guardar en DB
PARAMS = {
    'seed': SEED,
    'n': N,
    'ch': CH,
    'T': T_STEPS,
    'gamma': 0.7,
    'del_t': 0.9
}

# Procesar cada clase
start_time = time.time()

for clase in CLASES_A_PROCESAR:
    print(f"\n{'='*70}")
    print(f"PROCESANDO CLASE {clase}")
    print(f"{'='*70}")
    
    indices = imagenes_por_clase[clase]
    if LIMITE_POR_CLASE:
        indices = indices[:LIMITE_POR_CLASE]
    
    # Verificar imágenes ya procesadas
    processed_set = set()
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    try:
        cursor.execute(f'SELECT image_idx FROM clase_{clase}')
        processed_set = set(row[0] for row in cursor.fetchall())
    except:
        pass
    conn.close()
    
    indices_pendientes = [idx for idx in indices if idx not in processed_set]
    
    print(f"✅ Imágenes ya procesadas: {len(processed_set)}")
    print(f"⏳ Imágenes pendientes: {len(indices_pendientes)}")
    
    if len(indices_pendientes) == 0:
        print(f"✨ Clase {clase} ya está completamente procesada. Saltando...")
        continue
    
    # Barra de progreso
    for idx in tqdm(indices_pendientes, desc=f"Clase {clase}"):
        img, label = train_dataset[idx]
        
        # Calcular R vs C
        c_values, r_values = calcular_R_vs_C(
            img, kblock, x_init, DEVICE, c_range,
            ch=CH, h=IMG_SIZE, w=IMG_SIZE,
            T=T_STEPS, gamma=PARAMS['gamma'], del_t=PARAMS['del_t']
        )
        
        # Guardar en base de datos
        guardar_resultado_db(DB_PATH, clase, idx, c_values, r_values, PARAMS)

elapsed_total = time.time() - start_time
print("\n" + "="*70)
print("PROCESAMIENTO COMPLETADO")
print("="*70)
print(f"⏱️  Tiempo total: {elapsed_total/3600:.2f} horas")
print(f"📁 Base de datos: {DB_PATH}")

## 12. Visualización de Resultados

**Curvas R vs C promedio por clase**

In [ ]:
# Graficar curvas R vs C promedio por clase
fig, axes = plt.subplots(2, 5, figsize=(20, 10))
axes = axes.flatten()

for clase in CLASES_A_PROCESAR:
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    # Cargar todos los datos de la clase
    cursor.execute(f'SELECT c_values, r_values FROM clase_{clase}')
    rows = cursor.fetchall()
    conn.close()
    
    if len(rows) == 0:
        axes[clase].text(0.5, 0.5, 'Sin datos', ha='center', va='center')
        axes[clase].set_title(f"Clase {clase}")
        continue
    
    # Deserializar y promediar
    all_r = []
    for row in rows:
        c_vals = pickle.loads(row[0])
        r_vals = pickle.loads(row[1])
        all_r.append(r_vals)
    
    all_r = np.array(all_r)
    r_mean = np.mean(all_r, axis=0)
    r_std = np.std(all_r, axis=0)
    
    # Graficar
    axes[clase].plot(c_vals, r_mean, 'b-', linewidth=2, label='Media')
    axes[clase].fill_between(c_vals, r_mean - r_std, r_mean + r_std, 
                             alpha=0.3, label='±1 std')
    axes[clase].set_title(f"Clase {clase} (n={len(rows)})")
    axes[clase].set_xlabel("C (acoplamiento)")
    axes[clase].set_ylabel("R (parámetro de orden)")
    axes[clase].grid(True, alpha=0.3)
    axes[clase].legend(fontsize=8)
    axes[clase].set_ylim([0, 1])

plt.tight_layout()
plot_path = RESULTS_DIR / 'curvas_R_vs_C_por_clase.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Gráfico guardado: {plot_path}")

## 13. Comparación de Curvas entre Clases (Superpuestas)

In [ ]:
# Comparación de curvas R vs C entre todas las clases
plt.figure(figsize=(14, 8))

colores = plt.cm.tab10(np.linspace(0, 1, 10))

for clase in CLASES_A_PROCESAR:
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    cursor.execute(f'SELECT c_values, r_values FROM clase_{clase}')
    rows = cursor.fetchall()
    conn.close()
    
    if len(rows) == 0:
        continue
    
    # Deserializar y promediar
    all_r = []
    for row in rows:
        c_vals = pickle.loads(row[0])
        r_vals = pickle.loads(row[1])
        all_r.append(r_vals)
    
    r_mean = np.mean(all_r, axis=0)
    r_std = np.std(all_r, axis=0)
    
    # Graficar
    plt.plot(c_vals, r_mean, '-', linewidth=2, color=colores[clase], 
             label=f'Clase {clase} (n={len(rows)})')
    plt.fill_between(c_vals, r_mean - r_std, r_mean + r_std, 
                     alpha=0.1, color=colores[clase])

plt.xlabel('C (parámetro de acoplamiento)', fontsize=12)
plt.ylabel('R (parámetro de orden)', fontsize=12)
plt.title('Comparación de R vs C entre clases de MNIST', fontsize=14, fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
plt.grid(True, alpha=0.3)
plt.ylim([0, 1])
plt.tight_layout()

plot_path = RESULTS_DIR / 'comparacion_R_vs_C_todas_clases.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Gráfico guardado: {plot_path}")

## 14. Resumen Final

In [ ]:
# Resumen de datos guardados
print("\n" + "="*70)
print("RESUMEN FINAL")
print("="*70)

total_imagenes = 0

for clase in CLASES_A_PROCESAR:
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    try:
        cursor.execute(f'SELECT COUNT(*) FROM clase_{clase}')
        n_images = cursor.fetchone()[0]
        total_imagenes += n_images
        
        if n_images > 0:
            print(f"\nClase {clase}:")
            print(f"  Imágenes procesadas: {n_images:,}")
            print(f"  Datos por imagen: {N_C_POINTS} pares (C, R)")
            print(f"  Tamaño total clase: ~{n_images * N_C_POINTS * 8 * 2 / (1024*1024):.2f} MB")
    except:
        pass
    
    conn.close()

print(f"\n{'='*70}")
print(f"TOTAL: {total_imagenes:,} imágenes procesadas")
print(f"Base de datos: {DB_PATH}")
if DB_PATH.exists():
    print(f"Tamaño en disco: {DB_PATH.stat().st_size / (1024*1024):.2f} MB")
print(f"{'='*70}")

## 15. Verificación Final y Persistencia

**Interpretación de resultados:**
- **R(C)**: Parámetro de orden de Kuramoto en función del acoplamiento
- **Transición de fase**: Donde R aumenta rápidamente (sincronización)
- **Variabilidad por clase**: Diferentes topologías muestran diferentes curvas R(C)

In [ ]:
# Verificar persistencia de datos
if IN_COLAB:
    print("✅ Datos guardados en Google Drive:")
    print(f"   📁 {DB_PATH}")
    print(f"   💾 Tamaño: {DB_PATH.stat().st_size / (1024*1024):.2f} MB")
    print("\n🔄 Los datos persisten entre sesiones de Colab")
    print("   Puedes detener y reiniciar el notebook sin perder progreso")
    
    # Opción de descargar también
    try:
        from google.colab import files
        print("\n📥 ¿Deseas descargar la base de datos también?")
        print("   (Descomenta la siguiente línea para descargar)")
        # files.download(str(DB_PATH))
    except:
        pass
else:
    print("ℹ️  No estás en Colab, la base de datos está en:", DB_PATH)
    
print("\n🔐 REPRODUCIBILIDAD GARANTIZADA:")
print(f"   - Semilla: {SEED}")
print(f"   - x_init: MISMO para todas las imágenes")
print(f"   - Patrón: IDÉNTICO a calculo_alpha_c_mnist_autocontenido.ipynb")
print(f"\n📊 Base de datos: {DB_PATH}")
print(f"   - Accesible desde cualquier sesión de Colab")
print(f"   - Reinicia el notebook y continuará automáticamente")